# Pool kandidat + skor makna (pool-makna-v1)

Hasil diskusi `2026-11-22-kualitas-terjemahan-tingkat-tutur.md`. Model demo v2 (M2M100-418M, `m2m_base`) tidak dilatih ulang.

- **MODE = "valid"** (run A, kernel `mdevian/unggah-ungguh-pool-makna-v1`): pool beam8 dan beam16 (decode terpisah)
  untuk 175 pasangan valid, skor forward, reranker, juri valid, dan reverse jv→id. Melatih reverse 5 epoch. Uji 20 pasangan fakta.
- **MODE = "test"** (run B, kernel `mdevian/unggah-ungguh-pool-makna-v1-test`): dibuat otomatis oleh
  `scripts/pilih_pool_makna.py` setelah pilihan dikunci di valid. Menilai baseline dan pilihan di 176 test + 225 kontrol.

Test tidak disentuh di MODE valid.

In [ ]:
MODE = "valid"
PILIHAN = None  # diisi otomatis oleh scripts/pilih_pool_makna.py untuk MODE test

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"], check=False)

import glob, json, time, random, os, hashlib, io, platform
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

SEED, MAXLEN, LAM = 42, 64, 2.0
BEAMS = [8, 16]
REV_EPOCHS, REV_BS, REV_LR = 5, 16, 5e-5
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
MANIFEST = {"mode": MODE, "seed": SEED, "maxlen": MAXLEN, "lambda": LAM, "beams": BEAMS,
            "reverse": {"epochs": REV_EPOCHS, "bs": REV_BS, "lr": REV_LR},
            "versions": {"python": platform.python_version(), "torch": torch.__version__,
                         "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__},
            "gpu": torch.cuda.get_device_name(0), "gates": {}, "inputs": {}}


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


def save_manifest():
    MANIFEST["elapsed_s"] = round(time.time() - T0)
    json.dump(MANIFEST, open(f"{OUT}/manifest_{MODE}.json", "w"), indent=1, ensure_ascii=False)

## Preflight: input, model v2, split

In [ ]:
cfgs = glob.glob("/kaggle/input/**/model/demo_config.json", recursive=True)
assert len(cfgs) == 1, f"harus tepat satu model/demo_config.json, ketemu {cfgs}"
MODEL_DIR = os.path.dirname(cfgs[0]); V2_DIR = os.path.dirname(MODEL_DIR)
CFG = json.load(open(cfgs[0]))
assert CFG["variant"] == "m2m_base" and CFG["nbest"] == 8 and float(CFG["rerank_lambda"]) == LAM, CFG
MANIFEST["inputs"]["v2_dir"] = V2_DIR
MANIFEST["inputs"]["demo_config_sha256"] = sha256(cfgs[0])

def load3(n):
    ps = glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True)
    ps = [p for p in ps if not p.startswith(V2_DIR)]
    assert len(ps) == 1, f"{n}.csv: {ps}"
    MANIFEST["inputs"][f"{n}_csv_sha256"] = sha256(ps[0])
    d = pd.read_csv(ps[0]); d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (load3(n) for n in ["train", "valid", "test"])
assert (len(tr), len(va), len(te)) == (1395, 175, 176), (len(tr), len(va), len(te))
log("data", len(tr), len(va), len(te), MANIFEST["gpu"])
tag_src = lambda ind, level: f"{TAG[level]} {ind}"

tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang=CFG["src_lang"], tgt_lang=CFG["tgt_lang"])
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).float().to(dev).eval()
BOS = tok.get_lang_id(CFG["tgt_lang"])
rr_dir = os.path.join(MODEL_DIR, CFG["reranker_dir"])
rr_tok = AutoTokenizer.from_pretrained(rr_dir)
rr_mod = AutoModelForSequenceClassification.from_pretrained(rr_dir).float().to(dev).eval()
log("model v2 + reranker dimuat dari", MODEL_DIR)

## Fungsi skor (resep v2: tag, forced BOS, max 64, log-prob rata-rata teacher-forced)

In [ ]:
def seq_scores(tok_, model_, srcs, cands, chunk=16):
    out = []
    for i in range(0, len(srcs), chunk):
        enc = tok_(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                   padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok_.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model_(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out, dtype=np.float64)


def nbest(srcs, k, bs=8):
    """num_beams = num_return_sequences = k, persis seperti nbest() v2. Beam16 didekode terpisah dari beam8."""
    out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=BOS, num_beams=k, num_return_sequences=k, max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        for n, j in enumerate(idx):
            out[j] = txt[n * k:(n + 1) * k]
    torch.cuda.empty_cache()
    return out


def cls_logprobs(ctok, cmod, sents):
    out = []
    for i in range(0, len(sents), 64):
        enc = ctok(sents[i:i + 64], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad():
            out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
    return torch.cat(out).numpy().astype(np.float64)


def build_pool(split, rows):
    """rows: list of dict(source_id, index, group, requested, indonesian, reference). Kembalikan DataFrame kandidat."""
    srcs = [tag_src(r["indonesian"], r["requested"]) for r in rows]
    recs = []
    for k in BEAMS:
        nb = nbest(srcs, k)
        for r, cands in zip(rows, nb):
            for rank, c in enumerate(cands):
                recs.append({"split": split, **r, "beam_width": k, "beam_rank": rank, "candidate": c})
        log(split, f"beam{k}", len(nb), "sumber")
    d = pd.DataFrame(recs)
    # skor forward & reranker dihitung sekali per teks unik dalam sumber–tingkat yang sama (cache)
    u = d.drop_duplicates(["source_id", "candidate"])[["source_id", "requested", "indonesian", "candidate"]].reset_index(drop=True)
    u["forward_logp_mean"] = seq_scores(tok, model, [tag_src(s, l) for s, l in zip(u.indonesian, u.requested)], u.candidate.tolist())
    lp = cls_logprobs(rr_tok, rr_mod, u.candidate.tolist())
    for i, l in enumerate(LEVELS):
        u[f"rr_logp_{l}"] = lp[:, i]
    d = d.merge(u.drop(columns=["requested", "indonesian"]), on=["source_id", "candidate"], how="left", validate="many_to_one")
    log(split, "pool", len(d), "baris,", len(u), "teks unik")
    return d


def pick(d, mu=0.0, widths=(8,), rev_col="reverse_logp_mean"):
    """Pilih satu kandidat per source_id. Skor = forward + LAM*rr_logp[requested] + mu*reverse.
    Deduplikasi per sumber–tingkat, pemecah seri = urutan asal (beam_width, beam_rank)."""
    x = d[d.beam_width.isin(widths)].sort_values(["source_id", "beam_width", "beam_rank"])
    x = x.drop_duplicates(["source_id", "candidate"]).copy()
    rr = np.array([x[f"rr_logp_{l}"].iat[i] for i, l in enumerate(x.requested)])
    x["_s"] = x.forward_logp_mean.values + LAM * rr + (mu * x[rev_col].values if mu else 0.0)
    x["_o"] = np.arange(len(x))
    best = x.sort_values(["source_id", "_s", "_o"], ascending=[True, False, True]).drop_duplicates("source_id")
    return best.drop(columns=["_s", "_o"]).set_index("source_id")

## Model reverse jv→id (dipakai hanya bila pemenang memakai μ > 0)

In [ ]:
def train_reverse():
    seed_all(SEED)
    rtok = AutoTokenizer.from_pretrained("facebook/m2m100_418M", src_lang="jv", tgt_lang="id")
    rmod = AutoModelForSeq2SeqLM.from_pretrained("facebook/m2m100_418M").to(dev)
    pairs = tr.drop_duplicates(["javanese", "indonesian"])[["javanese", "indonesian"]].values.tolist()
    log("reverse: pasangan unik", len(pairs))
    nb = (len(pairs) + REV_BS - 1) // REV_BS
    opt = torch.optim.AdamW(rmod.parameters(), lr=REV_LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * REV_EPOCHS * nb), REV_EPOCHS * nb)
    scaler = torch.amp.GradScaler("cuda")
    for ep in range(REV_EPOCHS):
        rmod.train(); random.shuffle(pairs); tot = 0.0
        for i in range(0, len(pairs), REV_BS):
            b = pairs[i:i + REV_BS]
            enc = rtok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                       padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == rtok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                loss = rmod(**enc, labels=labels).loss
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(rmod.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(b)
        log("reverse epoch", ep + 1, "loss", round(tot / len(pairs), 4))
    rmod.eval()
    return rtok, rmod


def reverse_scores(rtok, rmod, jav, ind):
    """mean log P(sumber Indonesia | kandidat Jawa), teacher forcing."""
    return seq_scores(rtok, rmod, list(jav), list(ind))

## MODE valid (run A)

In [ ]:
FAKTA_CSV = r'''group,indonesian,setia,rusak,jenis_rusak,fakta_diubah
21,"Ketika berjalan-jalan nenek jatuh, dibantu berdiri oleh kakek.","Eyang putri tindak-tindak dhawah, dipunjumenengaken dening eyang kakung.","Eyang putri tindak-tindak dhahar, dipunjumenengaken dening eyang kakung.",predikat,jatuh -> makan
90,"Seandainya dinaiki mesti enak, sebab mobil saya itu baru.","Dipuntitihana mesti sakeca, amargi mobil adalem menika enggal.","Dipuntitihana mesti sakeca, amargi griya adalem menika enggal.",objek,mobil -> rumah
121,"Saya mempunyai mangga, apa kamu mau?","Kula gadhah pelem, menapa panjenengan kersa?","Kula gadhah sekar, menapa panjenengan kersa?",objek,mangga -> bunga
126,"Dia itu mau makan, maka badannya gemuk.","Panjenenganipun menika kersa dhahar, mila saliranipun sugeng.","Panjenenganipun menika boten kersa dhahar, mila saliranipun sugeng.",negasi,mau makan -> tidak mau makan
130,"Saya hanya dapat memberi jam ini, silakan pakai!","Adalem namung saged nyaosi jam menika, panjenengan agem!","Adalem namung saged nyaosi sabuk menika, panjenengan agem!",objek,jam -> ikat pinggang
133,Rumah Pak Purba yang tepi jalan sudah boleh saya beli.,Dalemipun Pak Purba ingkang pinggir margi sampun kepareng adalem tumbas.,Dalemipun Pak Purba ingkang pinggir margi sampun kepareng adalem sade.,predikat,beli -> jual
173,Bicaralah sendiri kepada ayah saya!,Panjenengan ngandika piyambak dhateng bapak adalem!,Panjenengan ngandika piyambak dhateng ibu adalem!,objek,ayah saya -> ibu saya
174,Ayahmu tadi berjalan-jalan dengan ayah saya.,Rama panjenengan (ingkang rama) wau tindak-tindak kaliyan bapak adalem.,Rama panjenengan (ingkang rama) wau dhahar kaliyan bapak adalem.,predikat,berjalan-jalan -> makan
192,Ibumu itu akan pergi ke mana?,Ibu panjenengan (ingkang ibu) menika badhe tindak dhateng pundi?,Ibu panjenengan (ingkang ibu) menika sampun tindak dhateng pundi?,waktu,akan pergi -> sudah pergi
207,"Temanmu sudah selesai buang air besar, sekarang sedang bercebok.","Kanca panjenengan sampun rampung bobotan, samenika saweg cawik.","Kanca panjenengan dereng rampung bobotan, samenika saweg cawik.",negasi,sudah selesai -> belum selesai
284,Itu anakmu akan dimandikan tidak mau.,Menika ingkang putra badhe dipun sirami boten kersa.,Menika ingkang rayi badhe dipun sirami boten kersa.,subjek,anakmu -> adikmu
351,Banyaklah yang melihat penobatan Raja di Majapahit.,Kathah ingkang sami mriksani jumenengan Ratu ing Majapait,Kathah ingkang sami mriksani jumenengan Ratu ing Mataram,objek,Majapahit -> Mataram
358,"Sudah Jeng Tanti jangan menangis, saya usap air matanya ya?!","Sampan Jeng Tanti sampun muwun, adalem lap waspanipun inggih?!","Sampan Jeng Tanti sampun muwun, adalem lap astanipun inggih?!",objek,air mata -> tangan
359,Pendaman ari-ari itu kalau malam kauberi lampu!,Pendheman tuntunan menika manawi dalu panjenengan paringi dilah!,Pendheman tuntunan menika manawi enjing panjenengan paringi dilah!,waktu,malam -> pagi
396,"Bawalah jaket ini, dapat untuk tutup dada!","Jaket menika panjenengan ampil, kenging kagem tutup jaja!","Jaket menika panjenengan ampil, kenging kagem tutup mustaka!",objek,dada -> kepala
464,Adikmu juga senang kalau bermain dengan nenek saya.,Ingkang rayi ugi seneng yen ameng-ameng kaliyan embah adalem.,Ingkang putra ugi seneng yen ameng-ameng kaliyan embah adalem.,subjek,adikmu -> anakmu
586,"Wah, kamu menerima kejutan, undian Simpedes menadapatkan sepeda motor.","Wah, panjenengan tampi kejotan, undian Simpedes angsal pit-montor.","Wah, panjenengan tampi kejotan, undian Simpedes angsal griya.",objek,sepeda motor -> rumah
682,Pak Hasan dan istrinya pergi ke Singapura.,Pak Hasan kaliyan garwanipun tindak dhateng Singapur.,Pak Hasan kaliyan putranipun tindak dhateng Singapur.,subjek,istrinya -> anaknya
693,Matamu kok merah apa kurang tidur?,Tingal panjenengan kok abrit menapa kirang sare?,Tingal panjenengan kok abrit menapa kirang dhahar?,predikat,kurang tidur -> kurang makan
1371,"Tegakkan imanmu, Tuhan Yang Mahakasih pasti akan memberi pertolongan!","Ingkang jejeg kapitadosan panjenengan, Gusti Inkgang Mahaasih mesthi badhe paring pitulungan!","Ingkang jejeg kapitadosan panjenengan, Gusti Inkgang Mahaasih mesthi boten badhe paring pitulungan!",negasi,pasti memberi -> pasti tidak memberi
'''
FAKTA_SHA256 = "f948ec8611c57f515e46e5a0160d8dc3b40dbcb9cbc0a3d85de2ae28746c3f83"

if MODE == "valid":
    assert hashlib.sha256(FAKTA_CSV.encode("utf-8")).hexdigest() == FAKTA_SHA256, "fakta_valid.csv berubah sejak dikunci"
    MANIFEST["inputs"]["fakta_valid_sha256"] = FAKTA_SHA256
    fakta = pd.read_csv(io.StringIO(FAKTA_CSV))
    assert len(fakta) == 20 and set(fakta.group) <= set(va.group)

    rows = [{"source_id": f"valid:{r.index}:{r.level}", "index": r.index, "group": r.group, "requested": r.level,
             "indonesian": r.indonesian, "reference": r.javanese} for r in va.itertuples()]
    pool = build_pool("valid", rows)

    # juri valid baru (DistilBERT, resep v2, seed dipasang sebelum inisialisasi). Sama untuk semua varian.
    seed_all(SEED)
    jtok = AutoTokenizer.from_pretrained("w11wo/javanese-distilbert-small")
    jmod = AutoModelForSequenceClassification.from_pretrained("w11wo/javanese-distilbert-small", num_labels=3).to(dev)
    opt = torch.optim.AdamW(jmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        jmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = jtok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = jmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    jmod.eval()
    acc_ref = float((cls_logprobs(jtok, jmod, va.javanese.tolist()).argmax(-1) == va.label.values).mean())
    MANIFEST["valid_judge_acc_on_refs"] = round(acc_ref, 4)
    u = pool.candidate.unique().tolist()
    lab = dict(zip(u, [LEVELS[i] for i in cls_logprobs(jtok, jmod, u).argmax(-1)]))
    pool["valid_judge_label"] = pool.candidate.map(lab)
    del jmod; torch.cuda.empty_cache()

    # baseline beam8/λ=2 harus mereproduksi v2 (chrF++ valid rerank2.0 = 62.66)
    base = pick(pool, 0.0, (8,))
    base = base.loc[[r["source_id"] for r in rows]]
    c_base = sacrebleu.corpus_chrf(base.candidate.tolist(), [base.reference.tolist()], word_order=2).score
    MANIFEST["gates"]["baseline_valid_chrF++"] = round(c_base, 2)
    MANIFEST["gates"]["baseline_v2_tercatat"] = 62.66
    log("baseline valid chrF++", round(c_base, 2), "(v2 mencatat 62.66)")

    # reverse: dilatih dan dihitung tanpa syarat; gerbang cakupan menentukan promosi, bukan komputasi
    del model; torch.cuda.empty_cache()
    rtok, rmod = train_reverse()
    uu = pool.drop_duplicates(["source_id", "candidate"])[["source_id", "indonesian", "candidate"]].copy()
    uu["reverse_logp_mean"] = reverse_scores(rtok, rmod, uu.candidate, uu.indonesian)
    pool = pool.merge(uu[["source_id", "candidate", "reverse_logp_mean"]], on=["source_id", "candidate"], how="left", validate="many_to_one")

    fakta["skor_setia"] = reverse_scores(rtok, rmod, fakta.setia, fakta.indonesian)
    fakta["skor_rusak"] = reverse_scores(rtok, rmod, fakta.rusak, fakta.indonesian)
    fakta["benar"] = fakta.skor_setia > fakta.skor_rusak  # seri dihitung gagal
    fakta.to_csv(f"{OUT}/fakta_skor.csv", index=False)
    MANIFEST["gates"]["uji_fakta_benar"] = int(fakta.benar.sum())
    MANIFEST["gates"]["uji_fakta_lolos"] = bool(fakta.benar.sum() >= 16)
    log("uji fakta reverse:", int(fakta.benar.sum()), "/ 20")

    rmod.half().save_pretrained(f"{OUT}/reverse"); rtok.save_pretrained(f"{OUT}/reverse")
    cols = ["split", "source_id", "index", "group", "requested", "indonesian", "reference", "beam_width", "beam_rank",
            "candidate", "forward_logp_mean", "rr_logp_ngoko", "rr_logp_krama", "rr_logp_krama_alus",
            "reverse_logp_mean", "valid_judge_label"]
    pool = pool.sort_values(["source_id", "beam_width", "beam_rank"])[cols]
    pool.to_csv(f"{OUT}/kandidat_valid.csv", index=False, float_format="%.10g")
    MANIFEST["outputs"] = {"kandidat_valid_sha256": sha256(f"{OUT}/kandidat_valid.csv"), "n_baris": len(pool)}
    save_manifest()
    log("SELESAI run A. Lanjutkan: python scripts/pilih_pool_makna.py seleksi")

## MODE test (run B): hanya setelah pilihan dikunci

In [ ]:
if MODE == "test":
    assert PILIHAN is not None, "PILIHAN belum dikunci"
    MANIFEST["pilihan"] = PILIHAN
    mu, widths = float(PILIHAN["mu"]), tuple(PILIHAN["widths"])
    BEAMS = sorted(set(widths) | {8})
    u_src = sorted(te["indonesian"].unique())
    sets = {
        "test": [{"source_id": f"test:{r.index}:{r.level}", "index": r.index, "group": r.group, "requested": r.level,
                  "indonesian": r.indonesian, "reference": r.javanese} for r in te.itertuples()],
        "control": [{"source_id": f"control:{i}:{l}", "index": None, "group": None, "requested": l,
                     "indonesian": s, "reference": None} for i, s in enumerate(u_src) for l in LEVELS],
    }
    assert len(sets["control"]) == 225
    pools = {k: build_pool(k, v) for k, v in sets.items()}
    if mu:
        del model; torch.cuda.empty_cache()
        revs = glob.glob("/kaggle/input/**/reverse/config.json", recursive=True)
        assert len(revs) == 1, revs
        rdir = os.path.dirname(revs[0]); MANIFEST["inputs"]["reverse_dir"] = rdir
        rtok = AutoTokenizer.from_pretrained(rdir, src_lang="jv", tgt_lang="id")
        rmod = AutoModelForSeq2SeqLM.from_pretrained(rdir).float().to(dev).eval()
        for k, d in pools.items():
            uu = d.drop_duplicates(["source_id", "candidate"])[["source_id", "indonesian", "candidate"]].copy()
            uu["reverse_logp_mean"] = reverse_scores(rtok, rmod, uu.candidate, uu.indonesian)
            pools[k] = d.merge(uu[["source_id", "candidate", "reverse_logp_mean"]], on=["source_id", "candidate"], how="left")
    else:
        for d in pools.values():
            d["reverse_logp_mean"] = np.nan
    for k, d in pools.items():
        d.to_csv(f"{OUT}/kandidat_{k}.csv", index=False, float_format="%.10g")

    for name, (m, w) in {"baseline": (0.0, (8,)), "pilihan": (mu, widths)}.items():
        ht = pick(pools["test"], m, w).loc[[r["source_id"] for r in sets["test"]]]
        hc = pick(pools["control"], m, w).loc[[r["source_id"] for r in sets["control"]]]
        pred = te.copy(); pred["hyp"] = ht.candidate.values
        pred.to_csv(f"{OUT}/pred_{name}.csv", index=False)
        pd.DataFrame({"indonesian": hc.indonesian.values, "requested": hc.requested.values,
                      "output": hc.candidate.values}).to_csv(f"{OUT}/control_{name}.csv", index=False)
        MANIFEST.setdefault("test_chrF++", {})[name] = {
            lv: round(sacrebleu.corpus_chrf(pred.hyp[pred.level == lv].tolist(), [pred.javanese[pred.level == lv].tolist()],
                                            word_order=2).score, 2) for lv in LEVELS}

    # reproduksi baseline terhadap CSV v2 (dicek setelah penguncian; selisih dilaporkan, bukan diperbaiki)
    v2 = pd.read_csv(os.path.join(V2_DIR, "pred_m2m_base_rerank2.0.csv"))
    b = pd.read_csv(f"{OUT}/pred_baseline.csv")
    MANIFEST["reproduksi_v2_identik"] = f"{int((v2.hyp.values == b.hyp.values).sum())}/{len(b)}"
    save_manifest()
    log("SELESAI run B", json.dumps(MANIFEST["test_chrF++"]), "reproduksi", MANIFEST["reproduksi_v2_identik"])